# Phase 3.3 — Collaborative Filtering

Build a sparse user-item interaction matrix, train a lightweight matrix-factorization baseline, generate personalized Top-K recommendations, and compare its test performance with the Phase 3.1 baselines.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_user_item.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation_interactions.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

K = 10
N_COMPONENTS = 32

print("Processed directory:", PROCESSED_DIR)


Processed directory: f:\annuspeaks.com\recommendation-system\data\processed


## 3.3.1 Sparse User-Item Interaction Matrix

Use the aggregated training user-item representation from Phase 2.4.

The matrix contains users as rows, products as columns, and total behavioral preference weight as the value.


In [2]:
# Load the aggregated training user-item data.

train_ui = pd.read_csv(
    TRAIN_PATH,
    usecols=["user_id", "item_id", "total_weight"]
)

validation = pd.read_csv(
    VALIDATION_PATH,
    usecols=["user_id", "item_id"]
)

test = pd.read_csv(
    TEST_PATH,
    usecols=["user_id", "item_id"]
)

user_ids = train_ui["user_id"].unique()
item_ids = train_ui["item_id"].unique()

user_to_index = {
    user_id: idx for idx, user_id in enumerate(user_ids)
}
item_to_index_cf = {
    item_id: idx for idx, item_id in enumerate(item_ids)
}

rows = train_ui["user_id"].map(user_to_index).to_numpy()
cols = train_ui["item_id"].map(item_to_index_cf).to_numpy()
values = train_ui["total_weight"].to_numpy(dtype=np.float32)

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32,
)

print("Matrix shape:", user_item_matrix.shape)
print("Non-zero interactions:", f"{user_item_matrix.nnz:,}")
print("Sparsity:", f"{1 - user_item_matrix.nnz / user_item_matrix.size:.6f}")


Matrix shape: (1407580, 228392)
Non-zero interactions: 1,939,777
Sparsity: 0.000000


## 3.3.2 Matrix-Factorization Baseline

Use truncated SVD as a practical matrix-factorization baseline over the sparse interaction matrix.

The factor count is intentionally modest so this remains a baseline rather than a heavy final model.


In [3]:
# Train matrix-factorization baseline.

n_components = min(
    N_COMPONENTS,
    min(user_item_matrix.shape) - 1
)

svd = TruncatedSVD(
    n_components=n_components,
    algorithm="randomized",
    n_iter=3,
    random_state=42,
)

user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

print("Latent dimensions:", n_components)
print("User factors:", user_factors.shape)
print("Item factors:", item_factors.shape)
print("Explained variance ratio:", f"{svd.explained_variance_ratio_.sum():.4f}")


Latent dimensions: 32
User factors: (1407580, 32)
Item factors: (228392, 32)
Explained variance ratio: 0.1451


## 3.3.3 Personalized Top-K Recommendations

Generate recommendations by scoring products against the user's latent factor vector.

Previously interacted products are excluded.


In [4]:
# Build a compact seen-item lookup for users.

seen_items = {}

for user_id, item_id in zip(
    train_ui["user_id"],
    train_ui["item_id"]
):
    seen_items.setdefault(user_id, set()).add(item_id)

index_to_user = np.asarray(user_ids)
index_to_item_cf = np.asarray(item_ids)

def collaborative_recommend(user_id, k=10):
    if user_id not in user_to_index:
        return []

    user_idx = user_to_index[user_id]
    scores = item_factors @ user_factors[user_idx]

    seen = seen_items.get(user_id, set())

    if seen:
        seen_indices = [
            item_to_index_cf[item_id]
            for item_id in seen
            if item_id in item_to_index_cf
        ]
        scores[seen_indices] = -np.inf

    candidate_count = min(k, len(index_to_item_cf))
    top_indices = np.argpartition(
        scores,
        -candidate_count
    )[-candidate_count:]

    top_indices = top_indices[
        np.argsort(scores[top_indices])[::-1]
    ]

    return [
        {
            "item_id": index_to_item_cf[idx],
            "score": float(scores[idx]),
        }
        for idx in top_indices
    ]

example_user = user_ids[0]

print("Example user:", example_user)
display(
    pd.DataFrame(
        collaborative_recommend(example_user, K)
    )
)


Example user: 0


,item_id,score
0,20388,0.000021
1,369447,0.000012
2,162139,0.000008
3,198209,0.000008
4,147684,0.000007
5,7943,0.000007
6,9877,0.000007
7,387697,0.000006
8,272770,0.000006
9,320130,0.000006


## 3.3.4 Evaluate Collaborative Filtering

Use the same deterministic 200-user test sample style used for the content-based baseline.

Only training data is used to build the model; the final test interaction remains unseen.


In [5]:
# Build held-out test targets.

test_targets = (
    test.groupby("user_id")["item_id"]
    .last()
    .to_dict()
)

MAX_EVAL_USERS = 200
eval_users = sorted(test_targets)[:MAX_EVAL_USERS]

print("Evaluation users:", len(eval_users))


Evaluation users: 200


In [6]:
# Vectorized batch evaluation to avoid scoring the full catalog
# separately for every user.

hits = 0
evaluated = 0

for start in range(0, len(eval_users), 50):
    batch_users = eval_users[start:start + 50]

    valid_users = [
        user_id
        for user_id in batch_users
        if user_id in user_to_index
    ]

    if not valid_users:
        continue

    user_indices = [
        user_to_index[user_id]
        for user_id in valid_users
    ]

    batch_scores = (
        user_factors[user_indices]
        @ item_factors.T
    )

    for row_idx, user_id in enumerate(valid_users):
        scores = batch_scores[row_idx]

        seen = seen_items.get(user_id, set())

        if seen:
            seen_indices = [
                item_to_index_cf[item_id]
                for item_id in seen
                if item_id in item_to_index_cf
            ]
            scores[seen_indices] = -np.inf

        top_indices = np.argpartition(
            scores,
            -K
        )[-K:]

        target = test_targets[user_id]

        if target in index_to_item_cf[top_indices]:
            hits += 1

        evaluated += 1

cf_hit_rate = (
    hits / evaluated
    if evaluated
    else 0.0
)

cf_result = pd.DataFrame([{
    "model": "Collaborative Filtering",
    "K": K,
    "evaluated_users": evaluated,
    "hits": hits,
    "HitRate@10": cf_hit_rate,
}])

display(cf_result)


,model,K,evaluated_users,hits,HitRate@10
0,Collaborative Filtering,10,200,1,0.005


## 3.3.5 Compare With Baselines

Compare the collaborative-filtering result with the Phase 3.1 baselines.

The comparison is directional because the content-based and collaborative evaluations use the same held-out test methodology and deterministic evaluation sample size.


In [7]:
# Recalculate Phase 3.1-style baseline scores on the same 200 test users.

train_popularity = (
    train_ui
    .groupby("item_id")["total_weight"]
    .sum()
    .sort_values(ascending=False)
)

popular_items = train_popularity.index.to_numpy()

popularity_hits = 0
similar_hits = 0

# Simple popularity comparison.
for user_id in eval_users:
    if test_targets[user_id] in popular_items[:K]:
        popularity_hits += 1

popularity_hit_rate = (
    popularity_hits / len(eval_users)
    if eval_users
    else 0.0
)

comparison = pd.DataFrame([
    {
        "model": "Popularity",
        "K": K,
        "evaluated_users": len(eval_users),
        "hits": popularity_hits,
        "HitRate@10": popularity_hit_rate,
    },
    {
        "model": "Collaborative Filtering",
        "K": K,
        "evaluated_users": evaluated,
        "hits": hits,
        "HitRate@10": cf_hit_rate,
    },
])

display(comparison)


,model,K,evaluated_users,hits,HitRate@10
0,Popularity,10,200,1,0.005
1,Collaborative Filtering,10,200,1,0.005


## Phase 3.3 Completion

- Sparse user-item interaction matrix built.
- Truncated-SVD matrix-factorization baseline trained.
- Personalized Top-K recommendations generated.
- Collaborative HitRate@10 evaluated.
- Collaborative performance compared with the popularity baseline.


In [8]:
# Final Phase 3.3 validation

assert user_item_matrix.nnz > 0
assert user_factors.shape[1] == item_factors.shape[1]
assert evaluated > 0
assert 0 <= cf_hit_rate <= 1
assert "Collaborative Filtering" in comparison["model"].values

print("Phase 3.3 validation: PASS")
print("Matrix:", user_item_matrix.shape)
print("Non-zero entries:", f"{user_item_matrix.nnz:,}")
print("Latent dimensions:", n_components)
print("Collaborative HitRate@10:", f"{cf_hit_rate:.4f}")


Phase 3.3 validation: PASS
Matrix: (1407580, 228392)
Non-zero entries: 1,939,777
Latent dimensions: 32
Collaborative HitRate@10: 0.0050
